In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.interpolate import griddata
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os

In [2]:
# Configuration parameters
mesh_size = 2.0
Spring_stiffness = 3000  # Fixed stiffness for this comparison
NORMAL_STRESSES = [2.5, 5.0, 7.5, 10.0, 12.5]

control_mode = 1  # Stress Control (PC mode)
CONTROL_MODES = ["NormalStressDC", "NormalStressPC"]
CONTROL_MODES_TITLE = ["Displacement Control", "Stress Control"]

# Rupture positions for animation frames
RUPTURE_POSITIONS = [5, 15, 25, 35, 45, 55, 65, 75, 80, 85, 90, 95, 100, 105, 110, 115]

# Create output folder for GIFs
BASE_FOLDER = f'./Friction-Coefficient'
GIF_FOLDER = os.path.join(BASE_FOLDER, 'NormalStressPCGIF')
os.makedirs(GIF_FOLDER, exist_ok=True)

# Setup colormap for different normal stresses
norm = mcolors.Normalize(vmin=min(NORMAL_STRESSES), vmax=max(NORMAL_STRESSES))
colormap = cm.coolwarm  # Good for stress comparison (blue to red)
sm = cm.ScalarMappable(norm=norm, cmap=colormap)
sm.set_array([])

# Storage for all data
all_frame_data = []

print(f"Collecting data for all normal stresses in {CONTROL_MODES[control_mode]} mode...")
print(f"Fixed Stiffness: {Spring_stiffness} MPa")

for RUPTURE_POSITION in RUPTURE_POSITIONS:
    frame_data = {
        'position': RUPTURE_POSITION,
        's12_data': [],
        'mu_data': []
    }
    
    for normal_stress in NORMAL_STRESSES:
        # Construct data path for each normal stress
        DATA_FOLDER = f'./{CONTROL_MODES[control_mode]}/{normal_stress}MPa/ShearFace'
        npz_file = os.path.join(DATA_FOLDER, f'ShearFace-{RUPTURE_POSITION}.npz')
        
        if not os.path.exists(npz_file):
            print(f"File {npz_file} not found, skipping...")
            continue
        
        print(f"Loading σn={normal_stress}MPa, Position={RUPTURE_POSITION}mm...")
        
        # Load data
        data = np.load(npz_file)
        x = data['x']
        y = data['y']
        z = data['z']
        s12 = data['s12']
        mu = data['mu']
        
        # === Process S12 data along midline ===
        # Find midline along z
        z_min, z_max = z.min(), z.max()
        z_mid = 0.5 * (z_min + z_max)
        z_span = z_max - z_min
        
        # Select points near z midline
        tol_z = 0.02 * z_span  # 2% tolerance
        mid_mask_z = np.abs(z - z_mid) <= tol_z
        
        # Bin along y for S12
        y_min, y_max = y.min(), y.max()
        NUM_Y_BINS = 100
        y_edges = np.linspace(y_min, y_max, NUM_Y_BINS + 1)
        y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
        
        s12_midline = np.full(NUM_Y_BINS, np.nan)
        
        if np.count_nonzero(mid_mask_z) > 0:
            counts_s12 = np.zeros(NUM_Y_BINS, dtype=int)
            for yy, ss in zip(y[mid_mask_z], s12[mid_mask_z]):
                if not np.isnan(ss):
                    bi = np.searchsorted(y_edges, yy, side='right') - 1
                    if 0 <= bi < NUM_Y_BINS:
                        if np.isnan(s12_midline[bi]):
                            s12_midline[bi] = 0.0
                        s12_midline[bi] += ss
                        counts_s12[bi] += 1
            good_s12 = counts_s12 > 0
            s12_midline[good_s12] /= counts_s12[good_s12]
        
        # === Process mu data along midline ===
        mu_midline = np.full(NUM_Y_BINS, np.nan)
        
        if np.count_nonzero(mid_mask_z) > 0:
            counts_mu = np.zeros(NUM_Y_BINS, dtype=int)
            for yy, mm in zip(y[mid_mask_z], mu[mid_mask_z]):
                if not np.isnan(mm):
                    bi = np.searchsorted(y_edges, yy, side='right') - 1
                    if 0 <= bi < NUM_Y_BINS:
                        if np.isnan(mu_midline[bi]):
                            mu_midline[bi] = 0.0
                        mu_midline[bi] += mm
                        counts_mu[bi] += 1
            good_mu = counts_mu > 0
            mu_midline[good_mu] /= counts_mu[good_mu]
        
        # Store processed data
        frame_data['s12_data'].append({
            'normal_stress': normal_stress,
            'y_centers': y_centers,
            's12_midline': s12_midline,
            'color': colormap(norm(normal_stress))
        })
        
        frame_data['mu_data'].append({
            'normal_stress': normal_stress,
            'y_centers': y_centers,
            'mu_midline': mu_midline,
            'color': colormap(norm(normal_stress))
        })
    
    if frame_data['s12_data'] and frame_data['mu_data']:
        all_frame_data.append(frame_data)

# === Create combined animation ===
print("\nCreating combined S12 and Mu animation for different normal stresses...")

if all_frame_data:
    # Find global limits for consistent axes
    all_s12_values = []
    all_mu_values = []
    y_global_min, y_global_max = float('inf'), float('-inf')
    
    for frame in all_frame_data:
        for s12_data in frame['s12_data']:
            valid_s12 = s12_data['s12_midline'][~np.isnan(s12_data['s12_midline'])]
            if len(valid_s12) > 0:
                all_s12_values.extend(valid_s12)
            y_global_min = min(y_global_min, s12_data['y_centers'].min())
            y_global_max = max(y_global_max, s12_data['y_centers'].max())
        
        for mu_data in frame['mu_data']:
            valid_mu = mu_data['mu_midline'][~np.isnan(mu_data['mu_midline'])]
            if len(valid_mu) > 0:
                all_mu_values.extend(valid_mu)
    
    # Set y-axis limits with some padding
    if all_s12_values:
        s12_min, s12_max = np.percentile(all_s12_values, [1, 99])
        s12_range = s12_max - s12_min
        s12_min -= 0.1 * s12_range
        s12_max += 0.1 * s12_range
    else:
        s12_min, s12_max = -10, 10
    
    if all_mu_values:
        mu_min, mu_max = np.percentile(all_mu_values, [1, 99])
        mu_range = mu_max - mu_min
        mu_min = max(0, mu_min - 0.1 * mu_range)  # Keep mu_min >= 0
        mu_max += 0.1 * mu_range
    else:
        mu_min, mu_max = 0, 1
    
    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), constrained_layout=True)
    
    # Add colorbar
    cbar = fig.colorbar(sm, ax=[ax1, ax2], aspect=30, pad=0.02)
    cbar.set_label('Normal Stress σn (MPa)', rotation=270, labelpad=25, fontsize=12)
    
    def animate(frame_idx):
        ax1.clear()
        ax2.clear()
        
        frame_data = all_frame_data[frame_idx]
        position = frame_data['position']
        
        # Plot S12 data (upper subplot)
        for s12_data in frame_data['s12_data']:
            valid_mask = ~np.isnan(s12_data['s12_midline'])
            if np.any(valid_mask):
                ax1.plot(s12_data['y_centers'][valid_mask], 
                        s12_data['s12_midline'][valid_mask],
                        'o-', linewidth=2.5, markersize=4,
                        color=s12_data['color'],
                        label=f"σn={s12_data['normal_stress']}MPa",
                        alpha=0.8)
        
        ax1.set_xlabel('Y Position (mm)', fontsize=11)
        ax1.set_ylabel('Shear Stress S12 (MPa)', fontsize=11)
        ax1.set_title(f'Shear Stress at Midline - {CONTROL_MODES_TITLE[control_mode]}', fontsize=12)
        ax1.set_xlim(y_global_min, y_global_max)
        ax1.set_ylim(s12_min, s12_max)
        ax1.grid(True, alpha=0.3)
        ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        
        # Add legend for reference (optional, since we have colorbar)
        # ax1.legend(loc='upper left', fontsize=9, ncol=2)
        
        # Plot Mu data (lower subplot)
        for mu_data in frame_data['mu_data']:
            valid_mask = ~np.isnan(mu_data['mu_midline'])
            if np.any(valid_mask):
                ax2.plot(mu_data['y_centers'][valid_mask], 
                        mu_data['mu_midline'][valid_mask],
                        'o-', linewidth=2.5, markersize=4,
                        color=mu_data['color'],
                        label=f"σn={mu_data['normal_stress']}MPa",
                        alpha=0.8)
        
        # Add reference lines for mu
        ax2.axhline(y=0.7, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='μ=0.7')
        ax2.axhline(y=0.5, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='μ=0.5')
        
        ax2.set_xlabel('Y Position (mm)', fontsize=11)
        ax2.set_ylabel('Friction Coefficient μ', fontsize=11)
        ax2.set_title(f'Friction Coefficient at Midline - {CONTROL_MODES_TITLE[control_mode]}', fontsize=12)
        ax2.set_xlim(y_global_min, y_global_max)
        ax2.set_ylim(mu_min, mu_max)
        ax2.grid(True, alpha=0.3)
        
        # Add legend showing reference lines
        # ax2.legend(loc='upper right', fontsize=9, ncol=3)
        
        # Add overall title with all parameters
        fig.suptitle(f'Rupture Position: {position}mm | E={Spring_stiffness}MPa | Mesh: {mesh_size}mm', 
                    fontsize=14, fontweight='bold')
        
        return ax1.lines + ax2.lines
    
    # Create animation
    anim = FuncAnimation(fig, animate, frames=len(all_frame_data), 
                        interval=600, blit=False, repeat=True)
    
    # Save animation
    gif_filename = f'NormalStress-PC-S12_Mu.gif'
    gif_path = os.path.join(GIF_FOLDER, gif_filename)
    anim.save(gif_path, writer=PillowWriter(fps=2), dpi=150)
    plt.close(fig)
    
    print(f"\nAnimation saved: {gif_path}")
    print(f"Processed {len(all_frame_data)} frames with {len(NORMAL_STRESSES)} normal stress values")
    print(f"Mode: {CONTROL_MODES_TITLE[control_mode]}")
    print(f"Fixed Stiffness: {Spring_stiffness} MPa")
    print(f"Normal Stresses compared: {NORMAL_STRESSES} MPa")
else:
    print("No data available for animation")

# === Optional: Create a summary plot showing all normal stresses at a specific position ===
print("\nCreating summary plot for middle position...")

SUMMARY_POSITION = 55  # Middle position for summary
summary_fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), constrained_layout=True)

for normal_stress in NORMAL_STRESSES:
    DATA_FOLDER = f'./{CONTROL_MODES[control_mode]}/{normal_stress}MPa/ShearFace'
    npz_file = os.path.join(DATA_FOLDER, f'ShearFace-{SUMMARY_POSITION}.npz')
    
    if os.path.exists(npz_file):
        data = np.load(npz_file)
        y = data['y']
        z = data['z']
        s12 = data['s12']
        mu = data['mu']
        
        # Process data (same as above)
        z_mid = 0.5 * (z.min() + z.max())
        tol_z = 0.02 * (z.max() - z.min())
        mid_mask_z = np.abs(z - z_mid) <= tol_z
        
        y_min, y_max = y.min(), y.max()
        NUM_Y_BINS = 100
        y_edges = np.linspace(y_min, y_max, NUM_Y_BINS + 1)
        y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
        
        # Process S12
        s12_midline = np.full(NUM_Y_BINS, np.nan)
        if np.count_nonzero(mid_mask_z) > 0:
            counts_s12 = np.zeros(NUM_Y_BINS, dtype=int)
            for yy, ss in zip(y[mid_mask_z], s12[mid_mask_z]):
                if not np.isnan(ss):
                    bi = np.searchsorted(y_edges, yy, side='right') - 1
                    if 0 <= bi < NUM_Y_BINS:
                        if np.isnan(s12_midline[bi]):
                            s12_midline[bi] = 0.0
                        s12_midline[bi] += ss
                        counts_s12[bi] += 1
            good_s12 = counts_s12 > 0
            s12_midline[good_s12] /= counts_s12[good_s12]
        
        # Process mu
        mu_midline = np.full(NUM_Y_BINS, np.nan)
        if np.count_nonzero(mid_mask_z) > 0:
            counts_mu = np.zeros(NUM_Y_BINS, dtype=int)
            for yy, mm in zip(y[mid_mask_z], mu[mid_mask_z]):
                if not np.isnan(mm):
                    bi = np.searchsorted(y_edges, yy, side='right') - 1
                    if 0 <= bi < NUM_Y_BINS:
                        if np.isnan(mu_midline[bi]):
                            mu_midline[bi] = 0.0
                        mu_midline[bi] += mm
                        counts_mu[bi] += 1
            good_mu = counts_mu > 0
            mu_midline[good_mu] /= counts_mu[good_mu]
        
        # Plot
        color = colormap(norm(normal_stress))
        valid_s12 = ~np.isnan(s12_midline)
        valid_mu = ~np.isnan(mu_midline)
        
        if np.any(valid_s12):
            ax1.plot(y_centers[valid_s12], s12_midline[valid_s12],
                    'o-', linewidth=2.5, markersize=4,
                    color=color, label=f'σn={normal_stress}MPa',
                    alpha=0.8)
        
        if np.any(valid_mu):
            ax2.plot(y_centers[valid_mu], mu_midline[valid_mu],
                    'o-', linewidth=2.5, markersize=4,
                    color=color, label=f'σn={normal_stress}MPa',
                    alpha=0.8)

# Format summary plot
ax1.set_xlabel('Y Position (mm)', fontsize=11)
ax1.set_ylabel('Shear Stress S12 (MPa)', fontsize=11)
ax1.set_title(f'Shear Stress Comparison - {CONTROL_MODES_TITLE[control_mode]}', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.legend(loc='best', fontsize=10)

ax2.axhline(y=0.7, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='μ=0.7')
ax2.axhline(y=0.5, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='μ=0.5')
ax2.set_xlabel('Y Position (mm)', fontsize=11)
ax2.set_ylabel('Friction Coefficient μ', fontsize=11)
ax2.set_title(f'Friction Coefficient Comparison - {CONTROL_MODES_TITLE[control_mode]}', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='best', fontsize=10, ncol=2)

summary_fig.suptitle(f'Normal Stress Comparison at Position={SUMMARY_POSITION}mm | E={Spring_stiffness}MPa | {CONTROL_MODES_TITLE[control_mode]}', fontsize=14, fontweight='bold')
summary_fig.colorbar(sm, ax=[ax1, ax2], aspect=30, pad=0.02, label='Normal Stress σn (MPa)')

summary_path = os.path.join(GIF_FOLDER, f'Summary_NormalStress_Comparison_Pos{SUMMARY_POSITION}.png')
summary_fig.savefig(summary_path, dpi=300, bbox_inches='tight')
plt.close(summary_fig)
print(f"Summary plot saved: {summary_path}")

print("\nAll processing complete!")

Fixed Stiffness: 3000 MPa
Loading σn=2.5MPa, Position=5mm...
Loading σn=5.0MPa, Position=5mm...
Loading σn=7.5MPa, Position=5mm...
Loading σn=10.0MPa, Position=5mm...
Loading σn=12.5MPa, Position=5mm...
Loading σn=2.5MPa, Position=15mm...
Loading σn=5.0MPa, Position=15mm...
Loading σn=7.5MPa, Position=15mm...
Loading σn=10.0MPa, Position=15mm...
Loading σn=12.5MPa, Position=15mm...
Loading σn=2.5MPa, Position=25mm...
Loading σn=5.0MPa, Position=25mm...
Loading σn=7.5MPa, Position=25mm...
Loading σn=10.0MPa, Position=25mm...
Loading σn=12.5MPa, Position=25mm...
Loading σn=2.5MPa, Position=35mm...
Loading σn=5.0MPa, Position=35mm...
Loading σn=7.5MPa, Position=35mm...
Loading σn=10.0MPa, Position=35mm...
Loading σn=12.5MPa, Position=35mm...
Loading σn=2.5MPa, Position=45mm...
Loading σn=5.0MPa, Position=45mm...
Loading σn=7.5MPa, Position=45mm...
Loading σn=10.0MPa, Position=45mm...
Loading σn=12.5MPa, Position=45mm...
Loading σn=2.5MPa, Position=55mm...
Loading σn=5.0MPa, Position=55mm.